# Orchestration & Lakeflow Jobs

Orchestration connects pipelines into production workflows. In this module you create multi-task Jobs with dependencies (DAG), configure triggers (CRON, File Arrival), set up retries and alerts, pass parameters between tasks (taskValues) and monitor runs with System Tables.

| Exam Domain | Weight |
|---|---|
| Working with Lakeflow Jobs | 16% |
| Troubleshooting, Monitoring, and Optimization (run history, monitoring) | 10% |

## Introduction to Lakeflow Jobs

Lakeflow Jobs (formerly Workflows) is a managed orchestration service.

| Scenario | Solution |
|----------|----------|
| ETL pipeline with multiple steps | Multi-task Job |
| Daily report at fixed time | Scheduled Job |
| Reaction to new files | File Arrival Trigger |
| ML training pipeline | Job with notebook tasks |
| Run Lakeflow Pipeline | Job with Pipeline task |

| Feature | Jobs | Lakeflow Pipelines |
|---------|------|-----|
| Orchestration | General | ETL only |
| Dependencies | Manual configuration | Automatic (DAG) |
| Data Quality | Custom code | Built-in expectations |
| Flexibility | High | Opinionated |

Best practice: use Lakeflow Pipelines for ETL, and Jobs to orchestrate the pipelines together with other tasks.



### Task Types in Lakeflow Jobs

Every task in a job has a type. Choose the type by what the task runs.

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| Notebook | Run a Databricks notebook | ETL logic, ML training |
| Pipeline | Run a Lakeflow Spark Declarative Pipeline | Streaming/batch pipelines |
| Dashboard | Refresh a published AI/BI dashboard and send its subscriptions | Fresh BI after the Gold load; needs a serverless or pro SQL warehouse |
| Python Script | Run a Python file | Utility scripts |
| SQL | Run a SQL query | DDL, reporting queries |
| JAR | Run a Java/Scala JAR | Legacy Spark jobs |
| Spark Submit | Submit a Spark application | Custom Spark apps |
| If/Else Condition | Branch based on condition | Conditional workflows |
| For Each | Iterate over a list | Parameterized batch runs |

A repair run reruns only the **failed and downstream tasks**. It skips the tasks that succeeded, which saves compute and time.

**Exam Note:** Know that repair runs skip tasks that already succeeded. If/Else and For Each make conditional and iterative workflows possible.

## Preparing Notebooks for Job

The demo job runs the 3 simple notebooks below. They are ready in your course folder as `materials/orchestration/task_01_validate`, `task_02_transform` and `task_03_report`; the next cells show their code. Copying the snippets into your own notebooks is optional, for example if you want to change them.

### Task 1: Validate Source

A job should stop early when the source has too little data. This task checks the row count against the `min_rows` threshold. It publishes the count to downstream tasks with `dbutils.jobs.taskValues.set()`.

In [0]:
# TASK 1: Validate Source Data
# The same code is in materials/orchestration/task_01_validate

# Parameters from Job
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("min_rows", "100")

source_table = dbutils.widgets.get("source_table")
min_rows = int(dbutils.widgets.get("min_rows"))

# Validation
df = spark.table(source_table)
row_count = df.count()

if row_count < min_rows:
    raise Exception(f"Validation FAILED: {row_count} rows < {min_rows} minimum")

# Share results with downstream tasks as task values.
# dbutils.notebook.exit() does not create task values: its string only goes to
# a dbutils.notebook.run() caller or the run output. Outside a job, set() does nothing.
dbutils.jobs.taskValues.set(key="row_count", value=row_count)
dbutils.jobs.taskValues.set(key="source_table", value=source_table)
print(f"Validation PASSED: {row_count} rows (task values: row_count, source_table)")


### Task 2: Transform Data

A downstream task often needs a result from the task before it. This task reads the result of the previous task through `taskValues` and adds three columns (duration, cost per mile, processing date). It publishes `rows_transformed` as a task value.

Run here, outside a job, the next code cell prints `Rows validated upstream: 0`, because `debugValue` is returned. In the job run, the `transform` task output shows the row count from `validate`.

**Task values: syntax reference**

| Operation | Syntax |
|-----------|--------|
| Set value (upstream task) | `dbutils.jobs.taskValues.set(key="row_count", value=1000)` |
| Get value (downstream task) | `val = dbutils.jobs.taskValues.get(taskKey="validate", key="row_count", default=0, debugValue=0)` |
| `debugValue` / `default` | `debugValue` is returned when the notebook runs **outside a job** (interactive); `default` when the key is missing in a job run |
| Reference in task settings | `{{tasks.validate.values.row_count}}` (e.g. If/else condition, For each inputs, task parameters) |
| Not a task value | `dbutils.notebook.exit("...")` returns a string to `dbutils.notebook.run()` or the run output only |
| Supported types | `str`, `int`, `float`, `bool`, JSON-serializable `dict`/`list` |
| Max value size | 48 KB per key |

In [0]:
# TASK 2: Transform Data
# The same code is in materials/orchestration/task_02_transform

from pyspark.sql import functions as F
from datetime import date

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")
dbutils.widgets.text("run_date", "")

source_table = dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or str(date.today())

# Read the task value published by the upstream "validate" task.
# debugValue -> used when running this notebook interactively (outside a job)
# default    -> used in a job run if the key was not set upstream
# ValueError -> raised in a job run that has no task "validate"
#               (e.g. this demo notebook run as one job)
# The except below exists only because this demo notebook also runs as a single-notebook job.
# task_02_transform does not need it.
try:
    rows_validated = dbutils.jobs.taskValues.get(
        taskKey="validate",
        key="row_count",
        default=0,
        debugValue=0
    )
except ValueError as e:
    if "Task key does not exist" not in str(e):
        raise
    rows_validated = 0
print(f"Rows validated upstream: {rows_validated}")

# Transformation
df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("trip_duration_minutes",
                F.round((F.col("tpep_dropoff_datetime").cast("long") -
                         F.col("tpep_pickup_datetime").cast("long")) / 60, 2))
    .withColumn("cost_per_mile",
                F.when(F.col("trip_distance") > 0,
                       F.round(F.col("fare_amount") / F.col("trip_distance"), 2))
                .otherwise(0))
    .withColumn("processing_date", F.lit(run_date))
)

row_count = df_transformed.count()
print(f"Transformed {row_count} rows")

display(df_transformed.select(
    "trip_distance", "fare_amount", "trip_duration_minutes", "cost_per_mile"
).limit(5))

# Publish result for downstream tasks
dbutils.jobs.taskValues.set(key="rows_transformed", value=row_count)

### Task 3: Generate Report

A report task runs last, when the data is ready. This task aggregates the metrics and shows a summary table (avg fare, avg distance, max fare). Total trips and revenue are not shown. They go to task values for the next tasks.

In [0]:
# TASK 3: Generate Report
# The same code is in materials/orchestration/task_03_report

from pyspark.sql import functions as F

# Parameters
dbutils.widgets.text("source_table", "samples.nyctaxi.trips")

source_table = dbutils.widgets.get("source_table")

# Aggregations
df = spark.table(source_table)

report = df.agg(
    F.count("*").alias("total_trips"),
    F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
    F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
    F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
    F.round(F.max("fare_amount"), 2).alias("max_fare")
)

# Daily report (total trips and revenue go to task values below)
display(report.select("avg_fare", "avg_distance", "max_fare"))

# Publish headline metrics as task values (visible in the run's task output)
metrics = report.first()
dbutils.jobs.taskValues.set(key="total_trips", value=int(metrics.total_trips))
dbutils.jobs.taskValues.set(key="total_revenue", value=float(metrics.total_revenue))


### Creating a Multi-task Job in the UI

Build the job in the Jobs UI from the three notebooks above. Each notebook becomes one task, and the dependencies make the DAG.

**Step 1: Create a new job**
- [ ] **Jobs & Pipelines** → **Create new** → **Job**
- [ ] Name: `Demo_ETL_Pipeline`

<img src="../../../assets/images/93c107ca21a54aab98249cf47db0337d.png" width="800">

- [ ] Compute: **Serverless** is already selected (performance optimized), so there is no cluster to create

> Genie Code may show suggestion cards with **Apply** on this page (for example a table update trigger). Do not click **Apply** here: it adds that trigger to the job.

**Step 2: Add Task 1 (Validate)**
- [ ] Task name: `validate`
- [ ] Type: Notebook
- [ ] Path: `<your course folder>/materials/orchestration/task_01_validate` (in the path dialog: Users → your e-mail → your course folder)
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

<img src="../../../assets/images/214b868309344df3a6e81f3cc2a84c13.png" width="800">

**Step 3: Add Task 2 (Transform)**
- [ ] Task name: `transform`
- [ ] Depends on: `validate`
- [ ] Path: `<your course folder>/materials/orchestration/task_02_transform`
- [ ] Parameters: `source_table` = `samples.nyctaxi.trips`

**Step 4: Add Task 3 (Report)**
- [ ] Task name: `report`
- [ ] Depends on: `transform`
- [ ] Path: `<your course folder>/materials/orchestration/task_03_report`

<img src="../../../assets/images/a3cc387f44de4247bf275bdbd38efb84.png" width="800">

**Step 5: Run Job**
- [ ] Click **Run now** three times. Each run shows three green tasks in about a minute, and the **Runs** tab gets one column per run. *Run History* below uses these runs.
- [ ] Show: DAG visualization
- [ ] Show: Task logs and output



## Medallion Pipeline Job: Ready-to-Use Config

The folder `materials/medallion/` has 6 production-ready notebooks that implement a full Medallion pipeline. This UI demo runs them as one job, in the order of the layers.

| Layer | Notebook | Input | Output |
|-------|----------|-------|--------|
| **Bronze** | `bronze_customers` | CSV files | `bronze.bronze_customers` |
| **Bronze** | `bronze_orders` | JSON files | `bronze.bronze_orders` |
| **Silver** | `silver_customers` | bronze_customers | `silver.silver_customers` |
| **Silver** | `silver_orders_cleaned` | bronze_orders | `silver.silver_orders_cleaned` |
| **Gold** | `gold_customer_orders_summary` | silver_customers + silver_orders_cleaned | `gold.gold_customer_orders_summary` |
| **Gold** | `gold_daily_orders` | silver_orders_cleaned | `gold.gold_daily_orders` |

DAG structure:
```
bronze_customers ──→ silver_customers ──────→ gold_customer_orders_summary
bronze_orders ────→ silver_orders_cleaned ─┤
                                           └→ gold_daily_orders
```

The definition below is YAML in the bundle resource format. Paste it in the Jobs UI YAML editor, or put it in a bundle's `resources/` folder and run `databricks bundle deploy`. The Deploy Checklist shows both ways.

### DABs YAML Configuration

This is the Declarative Automation Bundles (formerly Databricks Asset Bundles, DABs) YAML definition of the Medallion pipeline Job. It was exported from a working Databricks deployment.

> Before you deploy, update `notebook_path` and `parameters` to match your environment. No compute is declared, so the tasks run on **serverless jobs compute**.

```yaml
resources:
  jobs:
    job_medalion_run:
      name: job_medalion_run
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/bronze_customers
            source: WORKSPACE

        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/bronze_orders
            source: WORKSPACE

        - task_key: silver_customers
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/silver_customers
            source: WORKSPACE

        - task_key: silver_orders_cleaned
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/silver_orders_cleaned
            source: WORKSPACE

        - task_key: gold_customer_order_summary
          depends_on:
            - task_key: silver_customers
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/gold_customer_orders_summary
            source: WORKSPACE

        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/materials/medallion/gold_daily_orders
            source: WORKSPACE

      queue:
        enabled: true
      parameters:
        - name: catalog
          default: <YOUR_CATALOG>
        - name: source_path
          default: /Volumes/<YOUR_CATALOG>/default/datasets
```

| Element | Description |
|---------|-------------|
| *(no compute key)* | Tasks run on **serverless jobs compute**. On classic compute use `job_cluster_key` (job cluster). `existing_cluster_id` (all-purpose cluster) is for ad-hoc tests only |
| `depends_on` | Defines task dependencies, which make up the DAG |
| `parameters` | Job-level parameters passed to all notebooks via `dbutils.widgets` |
| `queue.enabled` | Queues runs if max concurrent reached |
| `source: WORKSPACE` | Notebook is in Workspace (not Repos) |

### Deploy Checklist

**Option A: YAML editor (UI)**
1. [ ] **Jobs & Pipelines** → **Create new** → **Job**
2. [ ] Next to **Run now** click the kebab menu **⋮** → **Edit as YAML**
3. [ ] Paste the `tasks:` / `parameters:` content of the YAML above (replace placeholders)
4. [ ] Save → Run now

**Option B: Declarative Automation Bundle (CLI)**
```bash
# Save the YAML above as resources/medallion_job.yml inside a bundle, then:
databricks bundle validate -t dev
databricks bundle deploy   -t dev
databricks bundle run      -t dev job_medalion_run
```
> `databricks jobs create --json @job.json` also exists, but it expects **Jobs API JSON**, not this YAML.

After the deployment, show participants:
- [ ] DAG visualization (fan-out at Bronze, fan-in at Gold)
- [ ] Job parameters (catalog, source_path)
- [ ] No trigger: run the job with **Run now**
- [ ] Run, then watch the layers run in order (Bronze → Silver → Gold)
- [ ] Repair Run: fail one task on purpose, then repair. The repair reruns only the failed and downstream tasks


## Triggers and Schedule

A trigger decides when a job starts. Lakeflow Jobs has time-based triggers (schedule), data-driven triggers (file arrival, table update) and continuous runs. You set them in the UI.

| Trigger Type | Usage | Example |
|---|---|---|
| **Scheduled** | Time-based: fixed schedule (CRON) | `0 0 2 * * ?` (daily at 2:00) |
| **File arrival** | Data-driven: new files in a storage location | New file in a UC Volume folder |
| **Table update** | Data-driven: one or more Unity Catalog tables change | New commits on `bronze.orders` |
| **Continuous** | Always running: a new run starts when the previous one ends | Streaming-like |

A job without a trigger runs only when you click **Run now** or call the API.

Use a schedule when the data arrives on a clock. Use a file arrival or table update trigger when it does not: the job then starts when the data is there.

**Exam Note:** Know CRON syntax, the three trigger types of the exam guide (scheduled, file arrival, table update) and when to choose time-based or data-driven.

<img src="../../../assets/images/nb08-job-triggers.png" alt="When a job starts: a time-based schedule with CRON and time zone, data-driven file arrival and table update triggers, and continuous runs; without a trigger only Run now or the API" width="800">

### Trigger Configuration Checklist

Use this checklist to show the trigger options in the Lakeflow Jobs UI, under **Schedules & Triggers** on the job page.

> Genie Code may show suggestion cards with **Apply** on this page (for example a table update trigger). Do not click **Apply** here: it adds that trigger to the job.

| Trigger Type | Usage | Example |
|---|---|---|
| **Scheduled** | Time-based: fixed schedule (CRON) | `0 0 2 * * ?` (daily at 2:00) |
| **File arrival** | Data-driven: new files in a storage location | New file in a UC Volume folder |
| **Table update** | Data-driven: one or more Unity Catalog tables change | New commits on `bronze.orders` |
| **Continuous** | Always running: a new run starts when the previous one ends | Streaming-like |

<img src="../../../assets/images/6e23746ce063491fa8afb3dea6268a1d.png" width="800">

**Scheduled trigger**
- [ ] Add trigger → Scheduled
- [ ] Cron expression: `0 0 2 * * ?` (daily at 2:00)
- [ ] Timezone: `Europe/Warsaw`
- [ ] Show: Preview next runs

<img src="../../../assets/images/cf3cfc85162a466eb77e15e20df5c15c.png" width="800">

**File arrival trigger** (optional)
- [ ] Add trigger → File arrival
- [ ] Set **Storage location** to a Unity Catalog Volume path, e.g. `/Volumes/<catalog>/default/datasets/landing/`
- [ ] Optional: under **Advanced**, set **Minimum time between triggers in seconds** and **Wait after last change in seconds**

**Table update trigger** (optional)
- [ ] Add trigger → Table update → **Tables**: `<catalog>.bronze.orders`
- [ ] With more than one table, set **Trigger when** to **Any table is updated** or **All tables are updated**
- [ ] Optional: under **Advanced**, the same two wait options as for file arrival

This is the same trigger in bundle YAML. Set `condition` explicitly: a trigger saved without it was stored as `ALL_UPDATED` (tested 2026-11-10).

```yaml
trigger:
  pause_status: UNPAUSED
  table_update:
    table_names:
      - <catalog>.bronze.orders
    condition: ANY_UPDATED              # or ALL_UPDATED
    min_time_between_triggers_seconds: 300
    wait_after_last_change_seconds: 60
```
### Useful CRON Expressions

Common CRON patterns for job schedules:

```
0 0 2 * * ?        # Daily at 2:00
0 0 * * * ?        # Every hour
0 0 9 ? * MON-FRI  # Mon-Fri at 9:00
0 0 0 1 * ?        # First day of month
0 */15 * * * ?     # Every 15 minutes
```

## Options, Retry and Alerting

You set run options and alerts in the job and task settings in the UI, on three levels:
- Task level: timeout, retries (count and delay)
- Job level: max concurrent runs, job timeout
- Notifications: email (on failure or success), webhooks (Slack/Teams via Destinations)

| Scenario | Retry? | Why |
|----------|--------|-----|
| Network timeout | Yes | Transient error |
| API rate limit | Yes | Transient error |
| Data quality issue | No | Retry won't fix data |
| Code bug | No | Retry won't fix code |

### Control Flow as Code: Retries, If/else, For each

This bundle YAML puts per-task retries and the If/else and For each task types from the table above into one compact job definition. The Jobs API JSON uses the same field names.

<img src="../../../assets/images/nb08-control-flow.png" alt="Control flow in one job: validate with retries sets row_count, the If/else task enough_rows checks it, and on true the For each task check_each_table runs check_one_table for each table" width="800">

```yaml
resources:
  jobs:
    control_flow_demo:
      name: control_flow_demo
      parameters:
        - name: tables
          default: '["samples.nyctaxi.trips", "samples.tpch.orders"]'
      tasks:
        - task_key: validate                      # sets task value row_count
          notebook_task:
            notebook_path: /Workspace/Users/<your-email>/<course-folder>/materials/orchestration/task_01_validate
          max_retries: 2                          # retry transient failures ...
          min_retry_interval_millis: 60000        # ... waiting >= 60 s between attempts
          retry_on_timeout: false

        - task_key: enough_rows                   # If/else condition task
          depends_on:
            - task_key: validate
          condition_task:
            op: GREATER_THAN                      # EQUAL_TO, NOT_EQUAL, GREATER_THAN(_OR_EQUAL), LESS_THAN(_OR_EQUAL)
            left: "{{tasks.validate.values.row_count}}"
            right: "100"

        - task_key: check_each_table              # For each task, runs only on the TRUE branch
          depends_on:
            - task_key: enough_rows
              outcome: "true"
          for_each_task:
            inputs: "{{job.parameters.tables}}"   # JSON array
            concurrency: 2                        # parallel iterations (default 1)
            task:
              task_key: check_one_table
              notebook_task:
                notebook_path: /Workspace/Users/<your-email>/<course-folder>/materials/orchestration/task_01_validate
                base_parameters:
                  source_table: "{{input}}"       # current element
```

> **Exam Note:** retries are set **per task** (`max_retries`, `min_retry_interval_millis`). Branching uses `condition_task` and `depends_on.outcome: "true"/"false"`. Looping uses `for_each_task` with `{{input}}`.

> Inside a For each iteration the Jobs API rejects task values ("setting task values is not supported for iterations"). `materials/orchestration/task_01_validate` catches that error and skips them; the snippet above does not, so use the file from `materials` in this job.


## Serverless Compute for Jobs

Since 2025, Databricks offers serverless compute as the default option for Lakeflow Jobs. You do not manage clusters: the compute scales automatically and exists only for the run. Serverless jobs use performance-optimized mode by default, which starts fast. Standard mode (`performance_target: STANDARD`) costs less, but takes 4–6 min to start. See [Select a performance mode](https://learn.microsoft.com/azure/databricks/jobs/run-serverless-jobs#select-a-performance-mode).

| Aspect              | Classic Clusters                  | Serverless Compute           |
|---------------------|----------------------------------|------------------------------|
| **Startup time**    | 3-10 minutes                     | Fast in performance-optimized mode (default); about 4–6 min in standard mode |
| **Cluster management** | Manual sizing (workers, instance types) | Fully automatic         |
| **Cost model**      | Per-VM (DBU + cloud infra)       | Per-DBU only (no infra cost) |
| **Spot instances**  | Configurable (cost savings)      | Not applicable               |
| **Auto-scaling**    | Configurable (min/max workers)   | Built-in, transparent        |
| **Custom libraries**| Full control (init scripts, pip) | Limited (pip only, no init scripts) |
| **GPU support**     | Yes                              | AI Runtime (Public Preview)  |
| **Use case**        | Long-running, GPU, custom libs, cost-sensitive batch | ETL, SQL, notebooks, triggered jobs |

To use it, select **Serverless** as the compute type in the Job task configuration instead of a cluster.

#### When to Use Classic Clusters Instead

| Scenario                | Why Classic?                                  |
|-------------------------|-----------------------------------------------|
| GPU / ML training       | Serverless GPU (AI Runtime) is still Public Preview |
| Network isolation (Private Link) | Classic clusters support custom VNet/VPC |
| Custom init scripts     | Serverless only supports pip packages         |
| Specific instance types needed | Serverless auto-selects instance types |
| Long-running batch (hours) | Spot instances on classic can be significantly cheaper |

> **Pro Tip:** For triggered jobs that need a fast response (file arrival, table update), keep serverless in the default performance-optimized mode. Standard mode can take 4–6 minutes to start. For cost-sensitive scheduled batch jobs running for hours, compare the cost of serverless with classic clusters on spot instances.

## Widgets and Parameters

Databricks Widgets let you parameterize notebooks. A job passes its parameters to the notebook through widgets, so one notebook can run for different environments or tables.

After you run the next cell, its widgets appear in the bar at the top of the notebook, next to the ones the task cells created.

In [0]:
# One widget of each type

# Text: any text
dbutils.widgets.text("environment", "dev", "Environment")

# Dropdown: select from a list
dbutils.widgets.dropdown("region", "EU", ["EU", "US", "APAC"], "Region")

# Combobox: dropdown where you can also type a value
dbutils.widgets.combobox("table", "orders", ["orders", "customers", "products"], "Table")

# Multiselect: select several values
dbutils.widgets.multiselect("columns", "id", ["id", "name", "date", "amount"], "Columns")

In [0]:
# Read the values (in a job run, the task parameters fill them)
environment = dbutils.widgets.get("environment")
region = dbutils.widgets.get("region")
table = dbutils.widgets.get("table")
columns = dbutils.widgets.get("columns")  # returns comma-separated string

print(f"environment={environment}, region={region}, table={table}, columns={columns}")

In [0]:
# Dynamic values for job and task parameters
# They are filled in only when the notebook runs as a task in a job
display(spark.createDataFrame([
    ("{{job.start_time.iso_date}}", "Run date (YYYY-MM-DD)"),
    ("{{job.start_time.iso_datetime}}", "Run start, date and time (UTC)"),
    ("{{job.id}}", "Job ID"),
    ("{{job.run_id}}", "Current job run ID"),
    ("{{task.run_id}}", "Current task run ID"),
    ("{{task.name}}", "Current task name"),
], "dynamic_value STRING, description STRING"))

In [0]:
# Remove the demo widgets so they do not stay on the notebook
dbutils.widgets.removeAll()

## Run History: Compare Against the Baseline

One slow run tells you little. The run history shows whether a job or one of its tasks is getting slower than its usual time (the baseline) and where a failed run stopped. Use it before you open any logs. Docs: [Monitor Lakeflow Jobs](https://learn.microsoft.com/azure/databricks/jobs/monitor).

Run `Demo_ETL_Pipeline` from the UI demo above two or three times first, so there is a history to compare.

1. Click **Jobs & Pipelines** in the sidebar and click the job name in the **Name** column. The **Runs** tab opens.
2. Look at the **matrix view**: one column per run, one row per task. The **Run total duration** row shows each run's duration and state. The bar height is the duration.
3. Hover over the `transform` cell of the latest run: start time, duration, status. Hover over the same task in the older runs. A task that takes clearly longer than in its earlier runs is the one to investigate, even if the run succeeded.
4. Read the colours: green succeeded, red failed, pink skipped, yellow waiting for retry, grey pending, canceled or timed out.
5. Click the link in the **Start time** column of a run. The **Job run details** page has **Graph**, **Timeline** and **List** views. In the **Graph** (the DAG), a failed task is red. The tasks after it show **Upstream failed**. They never ran, so the cause is in the red task.
6. Click the red task. **Task run details** shows the error message and the output. Read the first line of the error first.

> **Exam Note:** use the run history to compare execution times with earlier runs (the baseline). The DAG shows the upstream blocker. A job run is **Failed** when a leaf task failed and **Succeeded with failures** when only non-leaf tasks failed. The history is kept for 60 days.

## Repair Run: Rerun Only What Failed

When a task fails, you do not need to rerun the whole job. **Repair run** reruns the failed tasks and the tasks that depend on them. Tasks that succeeded keep their result. It works only for jobs with **two or more tasks**. For a single-task job, click **Run now** again. Docs: [Troubleshoot and repair job failures](https://learn.microsoft.com/azure/databricks/jobs/repair-job-failures).

<img src="../../../assets/images/nb08-repair-run.png" alt="Repair run of Demo_ETL_Pipeline: task_01_validate fails with min_rows = 999999999 and the next tasks are upstream failed; the repair with min_rows = 100 reruns all three in the same run" width="800">

Make `validate` fail on purpose with a job parameter:

1. Open `Demo_ETL_Pipeline` → in the **Job details** panel, **Job parameters** → **Edit parameters** → add `min_rows` = `999999999` → **Save**. Job parameters are passed to every task, so `task_01_validate` reads it from its widget.
2. Click **Run now**. `validate` fails, and `transform` and `report` show **Upstream failed**. On serverless, auto-optimization retries the failed task even without a retry policy, so the failure can take a moment (automatic retry). You can turn it off in the task's **Retries** settings (**Enable serverless auto-optimization**).
3. Open the failed run (link in **Start time**, or click it in the matrix view) and click **Repair run**. The **Repair job run** dialog lists the failed task and its dependents.
4. In the dialog, set the parameter `min_rows` = `100`. Values entered here override the job's values, and later repairs of this run keep them until you clear the key and value in the dialog.
5. Click **Repair run** in the dialog. `validate`, `transform` and `report` run again. In the matrix view the column of the repaired run turns green; the repair does not add a new column. The **Run parameters** column still shows `min_rows: 999999999` from the first attempt; the values the repair used are in **Task run details** of the repaired attempt.
6. Clean up: **Job parameters** → **Edit parameters** → remove `min_rows` → **Save**. Otherwise the next **Run now** fails again.

A repair reruns each task from the beginning. If a task appended part of its output before it failed, the repair can duplicate data, so write tasks idempotently (overwrite or `MERGE`, not append).

You can also repair a run from code. The CLI command reruns all failed tasks and their dependents:

```bash
databricks jobs repair-run <run_id> --rerun-all-failed-tasks
```

To pick the tasks by name instead, use `rerun_tasks` with `rerun_dependent_tasks`. The REST call `POST /api/2.2/jobs/runs/repair` can also pass new parameters, as in this request body:

```json
{"run_id": <run_id>, "rerun_all_failed_tasks": true, "job_parameters": {"min_rows": "100"}}
```

> **Exam Note:** repair reruns failed tasks **and their dependents**, skips the ones that succeeded, and accepts new parameters. The **Duration** of a repaired run counts from the first attempt to the end of the last repair.

## Monitoring via System Tables

> **Demo (trainer only).** System tables hold the job history of every workspace in the account region, including other teams' jobs and their `run_as`, so the course does not grant them to the training group. The trainer runs the two cells below. Participants: skip these two cells. Without grants on `system` they stop with a permission error.

Use system tables when you need the job history as data, for example to count failures per job or to track durations over days. The key tables are `system.lakeflow.job_run_timeline` (run history: start and end time, result state, trigger type) and `system.lakeflow.jobs` (job metadata, including the job **name**).

Before you query them:
- System tables are account-wide. They hold data from **every workspace in the account**, so always filter on `workspace_id` (a STRING column).
- Job names come from `system.lakeflow.jobs`. `run_name` in the timeline is set for one-time (submitted) runs, but it is NULL for runs of saved jobs. Join on `workspace_id` + `job_id` and keep the latest row per job (`ORDER BY change_time DESC`), because the jobs table keeps one row per change.
- One run is not always one row. A run longer than 1 hour is split into hourly rows, and `result_state` is set only on the final row. Aggregate per `run_id` first (`MIN` start, `MAX` end, `MAX_BY(result_state, period_end_time)` = final state) before you count runs or average durations.

In the second result, look at the `failed` column and compare `avg_duration_min` of a job across days.

In [0]:
from databricks.sdk import WorkspaceClient

# System tables are account-wide -> filter on THIS workspace (workspace_id is a STRING column)
ws_id = WorkspaceClient().get_workspace_id()
print(f"workspace_id = {ws_id}")

# Latest timeline rows in this workspace (a run longer than 1 h appears as several hourly rows)
spark.sql(f"""
    SELECT
        DATE(period_start_time) AS run_date,
        period_start_time       AS start_time,
        job_id,
        run_id,
        run_name,               -- NULL for runs of saved jobs (name: system.lakeflow.jobs)
        result_state,           -- set on the final row: SUCCEEDED, FAILED, ERROR, CANCELLED, ...
        trigger_type
    FROM system.lakeflow.job_run_timeline
    WHERE workspace_id = '{ws_id}'
    ORDER BY period_start_time DESC
    LIMIT 20
""").display()

In [0]:
# Per job, per day: runs, failed runs, average duration (last 7 days, this workspace)
# 1) runs: collapse the hourly timeline rows into ONE row per run_id
#    (MAX_BY = final result_state, so a repaired run counts once, with its final state)
# 2) jobs: latest name per job from system.lakeflow.jobs (one row per job change)
# Uses ws_id from the previous cell.
spark.sql(f"""
    WITH runs AS (
      SELECT workspace_id, job_id, run_id,
             MIN(period_start_time) AS start_time,
             MAX(period_end_time)   AS end_time,
             MAX_BY(result_state, period_end_time) AS result_state
      FROM system.lakeflow.job_run_timeline
      WHERE workspace_id = '{ws_id}' AND period_start_time >= current_date() - INTERVAL 7 DAYS
      GROUP BY ALL
    ),
    jobs AS (
      SELECT workspace_id, job_id, name,
             ROW_NUMBER() OVER (PARTITION BY workspace_id, job_id ORDER BY change_time DESC) AS rn
      FROM system.lakeflow.jobs
      WHERE workspace_id = '{ws_id}'
    )
    SELECT DATE(r.start_time) AS run_date,
           COALESCE(j.name, '(one-time run)') AS job_name,
           COUNT(*) AS runs,
           COUNT_IF(r.result_state IN ('FAILED','ERROR','TIMED_OUT')) AS failed,
           ROUND(AVG(TIMESTAMPDIFF(SECOND, r.start_time, r.end_time)) / 60, 1) AS avg_duration_min
    FROM runs r
    LEFT JOIN jobs j
      ON r.workspace_id = j.workspace_id AND r.job_id = j.job_id AND j.rn = 1
    GROUP BY ALL ORDER BY run_date DESC, runs DESC LIMIT 20
""").display()

## Summary

| Topic | Key Concept | Exam Keywords |
|---|---|---|
| **Multi-task Jobs** | DAG workflow, task dependencies | Task types, Repair Runs |
| **Triggers** | Scheduled (CRON), File arrival, Table update, Continuous | `0 0 2 * * ?`, `ANY_UPDATED` / `ALL_UPDATED` |
| **Options** | Timeout, Retry, Max concurrent runs | Transient vs permanent errors |
| **Alerting** | Email, Webhooks (Slack/Teams) | Notification destinations |
| **Parameters** | Widgets, dynamic values, taskValues | `dbutils.widgets`, `dbutils.jobs.taskValues` |
| **Monitoring** | System tables, success rate, duration | `system.lakeflow.job_run_timeline` |

> **← 07: Lakeflow Pipelines | Day 3 | 09: CI/CD & Bundles →**

← <a href="$../../day2/demo/07_lakeflow_pipelines">07 — Lakeflow Pipelines</a> | **<a href="$../../..">Course folder (README)</a>** | <a href="$./09_cicd_and_bundles">09 — CI/CD & Bundles →</a>

## Self-check (trainer): participants can skip this cell

In [0]:
# Self-check (trainer): the three task snippets produced their results (the demo writes no tables, no job runs here)

# Task 1: the source passed the min_rows threshold
assert row_count >= min_rows, (row_count, min_rows)

# Task 2: the transform added the three columns
missing = {"trip_duration_minutes", "cost_per_mile", "processing_date"} - set(df_transformed.columns)
assert not missing, f"missing columns: {missing}"

# Task 3: the report counts the same trips
assert metrics.total_trips == row_count, (metrics.total_trips, row_count)
assert metrics.avg_fare is not None, "the report has no avg_fare"

print("Self-check passed")